# Lecture 16 (Dynamic programming)

## Exercise 16.1 (coins)

In this exercise we would like to obtain some currency amount using the fewest coins available in the currency.  We assume that the set of different coin values of the currency is represented by a tuple, e.g. `coins = (1, 2, 5, 10, 20)` could represent the Danish coins.

In this exercise you can assume that all amounts and coins have integer values, and that there is coin of value 1 in the currency - this guarantees that all values can be expressed using the available coin values. Furthermore you might assume that all amounts are ≤ 200.

Consider the above `coins`. The amount 33 can be achieved in many different ways, e.g. 10 + 10 + 5 + 5 + 1 + 1 + 1 and 20 + 10 + 2 + 1. The last sum is a solution using the fewest coins.

A simple _greedy strategy_ to construct a list of coins for a given value, is to repeatedly take the coin with largest value until the remaining value is less that the value of the largest coin. Then repeatedly take the second largest valued coin as long as possible, etc. For some set of coins this strategy is guaranteed to always find an optimal solution, like for the above tuple `coins`. But in general this strategy will not always find the optimal strategy, e.g. for the value 14 and possible coins (1, 7, 10), the greedy strategy will find the solution 10 + 1 + 1 + 1 + 1, whereas an optimal solution using the fewest coins is 7 + 7.

1.  Implement a function `change_greedy(value, coins)` that implements the greedy change strategy, and returns a list of integers, each integer in `coins`, and with sum equal to `value`.

    _Example_. `change_greedy(35, (1, 7, 10))` should return `[10, 10, 10, 1, 1, 1, 1, 1]`.

2.  Implement a recursive function `number_of_coins(value, coins)` that returns the number of coins in an _optimal solution_. The function can implement the following recursive expression:

    *  If  _value_ = 0: _number\_of\_coins_(_value_, _coins_) = 0
    *  If  _value_ > 0: _number\_of\_coins_(_value_, _coins_) = 1 + min<sub>_coin_ in _coins_ where _coin_ ≤ _value_</sub> _number\_of\_coins_(_value_ - _coin_, _coins_)

    To speed up  your computation use memoization, e.g. using the decorator `@memoized` (without memoization, the second example below will run for a very very long time).

    _Example_. `number_of_coins(35, (1, 7, 10))` should return `5`, and `number_of_coins(100, (1, 2))` should return `50`.


3.  Implement a recursive function `change(value, coins)` that returns an optimal solution, i.e. a list of integers, each integer in `coins` and where the sum equals `value`.  One approach could be to modify your function `number_of_coins` appropriately to return a list of coins instead of a number.

    _Example_. `change(35, (1, 7, 10))` could return the list `[1, 7, 7, 10, 10]`.

The depth of the recursion when running the above recursive functions depends on the parameter `value`, and it is very likely you will get a `RecursionError: maximum recursion depth exceeded` when trying to compute `number_of_coins(1000, (1,))`. In the final question you should convert your `change` function into an iterative solution that systematically fills out the 'memoization table'.

4.  Implement a function `change_iterative(value, coins)` that returns an optimal solution. The function should fill out a table of solutions for increasing values of `value`.

    _Example_. `change_iterative(12345, [1, 2, 5, 10, 20])` could return `[20, 20, ..., 20, 20, 5]`.

In [ ]:
def memoize(f):
    answers = {}
    def wrapper(*args, **kwargs):
        if args not in answers:
            answers[args] = f(*args)
        return answers[args]
    wrapper.__name__ = f.__name__ + '_memoize'
    return wrapper

def number_of_coins(value, coins):
    if value == 0:
        return 0
    
    return 1 + min([number_of_coins(value - coin, coins) for coin in coins if coin <= value], key=len)

@memoize
def change(value, coins):
    if value == 0:
        return []
    
    return min([[coin] + change(value - coin, coins) for coin in coins if coin <= value], key=len)

print(change(14, (1, 7, 10)))

def change_iterative(value, coins):
    results = {}
    
    for v in range(1, value+1):
        for coin in coins: 
         

[7, 7]


## Exercise 16.2 (missing spaces) [optional]

Assume you have a text `text` where all spaces have been lost. To recover the spaces you have a dictionary `words` (a Python set) containing known words. The original text can also contain words not in the dictionary. The goal is to try to recover the spaces as good as possible: Find in the string non-overlapping occurrences of words from the dictionary, such that the total number of words identified plus the number of characters in the text not contained in any of the identified word occurrences is minimized.  Equivalently if we put '\_' between each word and character not in a word occurrence, then we would like to return a shortest such string, i.e. minimize the number of '\_' in the output.

    text = 'aduckcrossestheroad'
    words = {'a', 'ad', 'cross', 'duck', 'est', 'he', 'road', 'the'}
    solution_optimal = 'a_duck_cross_est_he_road'
    solution_greedy = 'ad_u_c_k_cross_est_he_road'

1.  Implement a _greedy strategy_ reading the text from left to right, always trying to find the longest possible word in the dictionary starting at the current position.


2.  Implement a dynamic programming solution, that finds an _optimal solution_. One systematic (non-recursive) approach is to create an array `solutions`, where you for increasing index `i`, from `0` to `len(text)`, compute an optimal solution for `text[:i]`     and store this in `solutions[i]`.


_Note_. As a dictionary of Danish words, you can try to download the list of [Danish words](https://dsn.dk/wp-content/uploads/2021/03/RO2012.opslagsord.med_.homnr_.og_.ordklasse.zip) from the webpage of [Dansk Sprognævn](https://dsn.dk/retskrivning/om-retskrivningsordbogen/ro-elektronisk-og-som-bog). Save the word list as `words.txt`, and run the below code to get the word list into your program as a set of words (in lower case without prefixes '1. ', '2. ', etc. and with spaces and periods removed).

    import re
    with open('words.txt', encoding='utf-8') as f:
         words = f.readlines()
    words = [w.split(';')[0] for w in words]
    words = {re.sub('^[1-9][.] |[ .]', '', w).lower() for w in words}

To get a sample text without spaces, your can e.g. use the following code.

    text = '''Danish multi line text sample, e.g. from https://da.wikipedia.org/wiki/Aarhus'''
    text = re.sub('[ \n,.]', '', text).lower()

## Exercise 16.3 - handin 8 (longest common subsequence)

A classic problem solvable using dynamic programming is the _longest common subsequence_ (LCS) problem: Given two strings `x` and `y`, find a longest possible string `z` that is a subsequence of both `x` and `y`.

_Example_. `LCS('**a**bracadab**ra**', '**a**z**ra**el')` is `'**a****ra**'`. Note that the result is not always unique, since both `'aba'` and `'aca'` are possible answers to `LCS('abca', 'acba')`.

The basic observation to solve this problem using dynamic programming is that:

* if `x == ''` or `y == ''`, then the longest common subsequence is `''`,

* if `x != ''` and `y != ''` and `x[-1] == y[-1]`, then there exist an optimal solution consisting of `LCS(x[:-1], y[:-1])` followed by `x[-1]`,

* if `x != ''` and `y != ''` and `x[-1] != y[-1]`, then a solution for `LCS(x, y)` is either `LCS(x[:-1], y)` or `LCS(x, y[:-1])`.


1.  Implement a _recursive_ function `lcs_length(x, y)` that computes the length of a longest common subsequence of `x`
    and `y`.  Use the `@memoize` decorator to speed up the recursive computation.


2.  Implement a _recursive_ function `lcs(x, y)` that computes a longest common subsequence of `x` and `y`.

In [2]:
import numpy as np
from scipy.integrate import quad

def f1(x):
    return 1/np.sqrt(x)

def f2(x):
    return np.log(x)/np.sqrt(x)

def f3(x):
    return np.exp(-x*x)

def f4(x):
    return 1/(1+x*x)

print(quad(f1, 0, 1))
print(quad(f2, 0, 1))
print(quad(f3, -np.inf, np.inf))
print(quad(f4, 0, np.inf))


(1.9999999999999984, 5.773159728050814e-15)
(-4.000000000000085, 1.354472090042691e-13)
(1.7724538509055159, 1.4202636756659625e-08)
(1.5707963267948966, 2.5777915205519274e-10)
